# Yazıcı Kullanım Analizi

> Yönetici özeti için sade ve sıralı analiz akışı.

Bu defter, `veriDepartment_Birlestirilmis.csv` dosyasını okur, temel kalite kontrolünü yapar ve karar vermeyi kolaylaştıran özet tabloları üretir. Hücreleri yukarıdan aşağıya çalıştırın.


## Akış

1. Veri dosyasının konumunu doğrula
2. Veriyi güvenli veri tipleriyle yükle ve düzenle
3. Kapsam ve veri kalitesini özetle
4. Aylık, departman ve kullanıcı seviyesinde sonuçları incele

`department_kaynak` alanı, ilgili departman değerinin hangi dosyadan geldiğini izlemek için korunur.


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd()
INPUT_FILE = PROJECT_DIR / "veriDepartment_Birlestirilmis.csv"

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Veri dosyası bulunamadı: {INPUT_FILE}\n"
        "Notebook'u yaziciProje klasöründen açın veya PROJECT_DIR değerini güncelleyin."
    )

print(f"Çalışma klasörü: {PROJECT_DIR}")
print(f"Kaynak veri: {INPUT_FILE.name}")


## 1. Veriyi yükle ve analiz sütunlarını hazırla

Bu hücre kaynak veriyi değiştirmez. Sayısal alanları sayısal türe çevirir ve renkli/siyah-beyaz/dijital kullanım toplamlarını üretir.


In [ ]:
def load_and_prepare(path: Path) -> pd.DataFrame:
    """Birleştirilmiş veriyi güvenli tiplerle yükler ve analiz sütunlarını üretir."""
    df = pd.read_csv(path, sep=";", encoding="cp1254", dtype={"ID": "string"})
    df.columns = df.columns.str.strip()
    df["Tarih"] = pd.to_datetime(df["Tarih"], dayfirst=True, errors="coerce")

    numeric_columns = [
        "renkli_tek_sayfa", "renkli_cift_sayfa", "sb_tek_sayfa", "sb_cift_sayfa",
        "kopya_renkli_tek", "kopya_renkli_cift", "kopya_sb_tek", "kopya_sb_cift",
        "tarama", "toplam",
    ]
    for column in numeric_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce").fillna(0)

    df["department"] = df["department"].fillna("").astype("string").str.strip()
    df["department_kaynak"] = df["department_kaynak"].fillna("").astype("string").str.strip()
    df["genel_renkli"] = (
        df["renkli_tek_sayfa"] + df["renkli_cift_sayfa"]
        + df["kopya_renkli_tek"] + df["kopya_renkli_cift"]
    )
    df["genel_sb"] = (
        df["sb_tek_sayfa"] + df["sb_cift_sayfa"]
        + df["kopya_sb_tek"] + df["kopya_sb_cift"]
    )
    df["dijital_islemler"] = df["tarama"]
    return df.sort_values(["Tarih", "ID"], na_position="last").reset_index(drop=True)


df = load_and_prepare(INPUT_FILE)
df.head()


## 2. Kapsam ve veri kalitesi

Boş departmanlar özellikle görünür bırakılır; böylece tamamlanacak kayıt miktarı net biçimde izlenir.


In [ ]:
quality_summary = pd.DataFrame({
    "Ölçüt": ["Kayıt", "Benzersiz çalışan", "Tarih aralığı", "Dolu department", "Boş department"],
    "Değer": [
        len(df),
        df["ID"].nunique(),
        f"{df['Tarih'].min():%d.%m.%Y} – {df['Tarih'].max():%d.%m.%Y}",
        int(df["department"].ne("").sum()),
        int(df["department"].eq("").sum()),
    ],
})
quality_summary


## 3. Yönetici özeti

Toplam hacim ve işlem türlerinin dağılımı, tek bir tabloda gösterilir.


In [ ]:
executive_summary = pd.DataFrame({
    "Metrik": ["Toplam sayfa", "Renkli işlem", "Siyah-beyaz işlem", "Dijital işlem"],
    "Değer": [
        df["toplam"].sum(),
        df["genel_renkli"].sum(),
        df["genel_sb"].sum(),
        df["dijital_islemler"].sum(),
    ],
})
executive_summary


## 4. Aylık kullanım

En son dönemler üstte olacak şekilde aylık toplamlar listelenir.


In [ ]:
monthly_summary = (
    df.dropna(subset=["Tarih"])
      .groupby(df["Tarih"].dt.to_period("M"))["toplam"]
      .sum()
      .rename("Toplam sayfa")
      .sort_index(ascending=False)
      .to_frame()
)
monthly_summary.head(12)


## 5. Departman görünümü

Boş departmanlı kayıtlar `Atanmamış` olarak ayrı gösterilir. Bu sayede sonuçlar ile veri tamamlama ihtiyacı birbirine karışmaz.


In [ ]:
department_summary = (
    df.assign(department_gorunum=df["department"].mask(df["department"].eq(""), "Atanmamış"))
      .groupby("department_gorunum")
      .agg(kayıt=("ID", "size"), çalışan=("ID", "nunique"), toplam_sayfa=("toplam", "sum"))
      .sort_values("toplam_sayfa", ascending=False)
)
department_summary


## 6. En yüksek kullanım gösteren çalışanlar

Bu liste yönetici için özet düzeydedir. Tek kişi veya dönem bazlı ayrıntılı incelemeler playground defterindedir.


In [ ]:
top_users = (
    df.groupby(["ID", "isim_soyisim"], dropna=False)["toplam"]
      .sum()
      .sort_values(ascending=False)
      .head(15)
      .rename("Toplam sayfa")
      .to_frame()
)
top_users


## Ek: Ham raporları birleştirme

Bu yardımcı fonksiyon, eski projedeki ham CSV raporlarını birleştirme adımını korur. Tek başına çalıştırıldığında veri okumaz; yalnızca gerektiğinde kullanılacak fonksiyonu tanımlar.


In [ ]:
def load_raw_reports(reports_dir: Path) -> pd.DataFrame:
    """Ham günlük yazıcı raporlarını tek tabloya birleştirir.

    Bu fonksiyon yalnızca ham raporlar yeniden işlenecekse çağrılmalıdır.
    """
    reports = sorted(reports_dir.glob("*.csv"))
    if not reports:
        raise FileNotFoundError(f"CSV raporu bulunamadı: {reports_dir}")

    frames = []
    for report in reports:
        lines = report.read_text(encoding="utf-8-sig").splitlines()
        header_row = next(
            index for index, line in enumerate(lines)
            if "Etkin Alan" in line and "Kullanıcı kimliği" in line
        )
        frame = pd.read_csv(report, skiprows=header_row, encoding="utf-8-sig")
        frame["Tarih"] = pd.to_datetime(report.stem[-10:], format="%Y-%m-%d", errors="raise")
        frames.append(frame)

    combined = pd.concat(frames, ignore_index=True).sort_values("Tarih").reset_index(drop=True)
    return combined


# Örnek kullanım (gerektiğinde yorum işaretini kaldırın):
# ham_veri = load_raw_reports(PROJECT_DIR / "reports")
